# TB vs Normal chest X-ray classifier: fixed InceptionV3 pipeline

What changed compared with the previous notebook:

1. **Correct input scaling.** ImageNet InceptionV3 expects pixels in [-1, 1]. The old cells fed [0, 1], so the frozen
   backbone was producing off-distribution features. The scaling now lives *inside* the model, so training and
   inference can never disagree.
2. **Native 299x299 input** for InceptionV3 (was 224).
3. **Deeper fine-tuning.** The top 50% of the backbone is unfrozen (BatchNorm stays frozen), with a cosine LR schedule,
   instead of only the last 30 layers at 1e-5.
4. **Both datasets are actually used.** Previously `combined_df` was built but training still used only the 492
   Shenzhen training images. Each image keeps a `source` column, and splits are stratified by label x source.
5. **5-fold cross-validation** instead of one 106-image test set, with a bootstrap confidence interval on AUC.
6. **Threshold chosen on validation data** for a target sensitivity (default 90%), not hand-picked per cell.
7. **Cross-source test** (train on one dataset, test on the other) to show how well the model generalises.
8. Grad-CAM, a 5-model ensemble for inference, and an optional TorchXRayVision baseline for comparison.

In [ ]:
import os, gc, glob, hashlib, io
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from IPython.display import display

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## Config

In [ ]:
SHENZHEN_DIR = "/kaggle/input/datasets/iamtapendu/chest-x-ray-lungs-segmentation"
SECOND_DIR = "/kaggle/input/datasets/yasserhessein/tuberculosis-chest-x-rays-images/Dataset of Tuberculosis Chest X-rays Images"
USE_SECOND_DATASET = True     # False = Shenzhen only
BALANCE_SECOND_DATASET = True # downsample the larger class of the 2nd dataset to match the smaller one

BACKBONE = "inceptionv3"      # "inceptionv3" | "densenet121" | "efficientnetv2b0"
PRETRAINED_WEIGHTS = "imagenet"

N_FOLDS = 5
INNER_VAL_FRACTION = 0.15     # carved from the training folds for early stopping + threshold choice
SEED = 42
BATCH_SIZE = 16
HEAD_EPOCHS = 8               # stage 1: frozen backbone, train the head only
FT_EPOCHS = 30                # stage 2: fine-tune (early stopping usually ends it sooner)
HEAD_LR = 1e-3
FT_LR = 3e-5
UNFREEZE_FRACTION = 0.5       # fraction of backbone layers (from the top) to unfreeze in stage 2
LABEL_SMOOTHING = 0.05
PATIENCE = 6
TARGET_SENSITIVITY = 0.90     # threshold is picked on validation data to reach this sensitivity
RUN_CROSS_SOURCE = True       # train on one source, test on the other (needs USE_SECOND_DATASET)
VERBOSE = 2
OUT_DIR = "/kaggle/working/tb_models"

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp")
os.makedirs(OUT_DIR, exist_ok=True)
keras.utils.set_random_seed(SEED)

if tf.config.list_physical_devices("GPU"):
    keras.mixed_precision.set_global_policy("mixed_float16")  # ~2x faster on T4/P100

BACKBONES = {
    # prep: how the backbone expects its input, starting from raw 0-255 pixels
    "inceptionv3":      dict(fn=keras.applications.InceptionV3,      size=299, prep="tf",    last_conv="mixed10"),
    "densenet121":      dict(fn=keras.applications.DenseNet121,      size=224, prep="torch", last_conv="relu"),
    "efficientnetv2b0": dict(fn=keras.applications.EfficientNetV2B0, size=224, prep="none",  last_conv="top_activation"),
}
CFG = BACKBONES[BACKBONE]
IMG_SIZE = CFG["size"]

## Load the file lists

In [ ]:
def list_shenzhen():
    meta = pd.read_csv(os.path.join(SHENZHEN_DIR, "MetaData.csv"))
    img_dir = os.path.join(SHENZHEN_DIR, "Chest-X-Ray", "Chest-X-Ray", "image")
    lookup = {os.path.splitext(f)[0]: os.path.join(r, f)
              for r, _, fs in os.walk(img_dir) for f in fs if f.lower().endswith(IMG_EXT)}
    out = pd.DataFrame({
        "image_path": meta["id"].astype(str).map(lookup),
        "label": meta["ptb"].astype(int),
        "source": "shenzhen",
    })
    missing = out["image_path"].isna().sum()
    if missing:
        print(f"Shenzhen: {missing} metadata rows had no image, dropped")
    return out.dropna(subset=["image_path"])


def list_second():
    rows = []
    for sub, label in [("TB Chest X-rays", 1), ("Normal Chest X-rays", 0)]:
        d = os.path.join(SECOND_DIR, sub)
        for f in sorted(os.listdir(d)):
            if f.lower().endswith(IMG_EXT):
                rows.append({"image_path": os.path.join(d, f), "label": label, "source": "dataset2"})
    out = pd.DataFrame(rows)
    if BALANCE_SECOND_DATASET:
        n = out["label"].value_counts().min()
        out = pd.concat([g.sample(n, random_state=SEED) for _, g in out.groupby("label")])
    return out


parts = [list_shenzhen()]
if USE_SECOND_DATASET:
    parts.append(list_second())
df = pd.concat(parts, ignore_index=True)

# Exact-duplicate check (same file bytes). Duplicates across folds would leak test images into training.
df["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in df["image_path"]]
dups = df["md5"].duplicated(keep="first")
if dups.any():
    print(f"Dropping {dups.sum()} exact duplicate images")
    conflicting = df[df["md5"].duplicated(keep=False)].groupby("md5")["label"].nunique().gt(1).sum()
    if conflicting:
        print(f"WARNING: {conflicting} duplicated images carry DIFFERENT labels")
df = df[~dups].reset_index(drop=True)

print(pd.crosstab(df["source"], df["label"].map({0: "Normal", 1: "TB"}), margins=True))

## Load all images into memory once (grayscale, uint8)
Around 1.7k images at 299x299 is about 150 MB, which is much faster than decoding PNGs every epoch for every fold.

In [ ]:
def load_gray(path_or_bytes, size=IMG_SIZE):
    if isinstance(path_or_bytes, (bytes, bytearray)):
        img = cv2.imdecode(np.frombuffer(path_or_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)
    else:
        img = cv2.imread(path_or_bytes, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image: {path_or_bytes if isinstance(path_or_bytes, str) else '<bytes>'}")
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)


X = np.stack([load_gray(p) for p in df["image_path"]])  # (N, H, W) uint8
y = df["label"].to_numpy().astype(np.int64)
print("X:", X.shape, X.dtype, "| y:", np.bincount(y))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.ravel(), np.random.default_rng(SEED).choice(len(df), 8, replace=False)):
    ax.imshow(X[i], cmap="gray")
    ax.set_title(f"{df.source[i]} | {'TB' if y[i] else 'Normal'}", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## Data pipeline and model
The model takes **raw 0-255 grayscale-as-RGB pixels**. Augmentation and backbone-specific scaling are layers inside
the model, so any image you pass at inference only needs resizing.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def to_rgb(img, label):
    img = tf.cast(img[..., None], tf.float32)
    return tf.repeat(img, 3, axis=-1), label


def make_ds(idx, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X[idx], y[idx].astype("float32")))
    if shuffle:
        ds = ds.shuffle(len(idx), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(to_rgb, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)


def make_augmenter():
    # Chest X-rays have a fixed orientation, so no flips. Geometry and intensity changes stay mild.
    return keras.Sequential([
        layers.RandomRotation(0.028, fill_mode="constant"),               # about +/-10 degrees
        layers.RandomZoom((-0.1, 0.1), fill_mode="constant"),
        layers.RandomTranslation(0.05, 0.05, fill_mode="constant"),
        layers.RandomContrast(0.15),
        layers.RandomBrightness(0.1, value_range=(0, 255)),
    ], name="augment")


def make_preprocess(kind):
    if kind == "tf":      # InceptionV3 / Xception / ResNetV2: [-1, 1]
        return layers.Rescaling(1 / 127.5, offset=-1.0, name="preprocess")
    if kind == "torch":   # DenseNet: ImageNet mean/std on [0, 1]
        return keras.Sequential([
            layers.Rescaling(1 / 255.0),
            layers.Normalization(mean=[0.485, 0.456, 0.406], variance=[0.229**2, 0.224**2, 0.225**2]),
        ], name="preprocess")
    return layers.Identity(name="preprocess")  # EfficientNetV2 rescales internally


def build_model():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = make_augmenter()(inp)
    x = layers.ReLU(max_value=255.0, name="clip")(x)  # keeps pixels in [0, 255] after augmentation
    x = make_preprocess(CFG["prep"])(x)
    backbone = CFG["fn"](include_top=False, weights=PRETRAINED_WEIGHTS, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    x = backbone(x, training=False)  # BatchNorm always in inference mode, which is safer with small batches
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    out = layers.Dense(1, activation="sigmoid", dtype="float32", name="head")(x)
    return keras.Model(inp, out, name=f"tb_{BACKBONE}"), backbone


def set_finetune(backbone, fraction):
    backbone.trainable = True
    cut = int(len(backbone.layers) * (1 - fraction))
    for i, layer in enumerate(backbone.layers):
        layer.trainable = i >= cut and not isinstance(layer, layers.BatchNormalization)


def train_model(tr_idx, va_idx):
    keras.backend.clear_session(); gc.collect()
    model, backbone = build_model()
    train_ds, val_ds = make_ds(tr_idx, shuffle=True), make_ds(va_idx)
    loss = keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING)

    # Stage 1: head only
    backbone.trainable = False
    model.compile(keras.optimizers.Adam(HEAD_LR), loss, metrics=[keras.metrics.AUC(name="auc")])
    model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, verbose=VERBOSE)

    # Stage 2: unfreeze the top of the backbone, cosine-decayed LR
    set_finetune(backbone, UNFREEZE_FRACTION)
    sched = keras.optimizers.schedules.CosineDecay(FT_LR, decay_steps=FT_EPOCHS * len(train_ds), alpha=0.05)
    model.compile(keras.optimizers.AdamW(sched, weight_decay=1e-4), loss,
                  metrics=[keras.metrics.AUC(name="auc")])
    hist = model.fit(
        train_ds, validation_data=val_ds, epochs=FT_EPOCHS, verbose=VERBOSE,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=PATIENCE,
                                                 restore_best_weights=True)])
    return model, hist


def predict(model, idx):
    return model.predict(make_ds(idx), verbose=0).ravel()

## Metrics helpers

In [ ]:
def pick_threshold(y_true, prob, target=TARGET_SENSITIVITY):
    """Highest threshold that still reaches the target sensitivity, which gives the best specificity at that sensitivity."""
    fpr, tpr, thr = roc_curve(y_true, prob)
    ok = np.where(tpr >= target)[0]
    return float(min(thr[ok[0]], 1.0)) if len(ok) else 0.5


def summarize(y_true, prob, thr):
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return dict(
        n=len(y_true),
        auc=roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan,
        sensitivity=float(tp / max(tp + fn, 1)),
        specificity=float(tn / max(tn + fp, 1)),
        accuracy=float((tp + tn) / len(y_true)),
        threshold=thr,
    )


def bootstrap_auc_ci(y_true, prob, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[i])) == 2:
            aucs.append(roc_auc_score(y_true[i], prob[i]))
    return np.percentile(aucs, [2.5, 97.5])

## 5-fold cross-validation
Each fold trains on 4/5 of the data (minus an inner validation slice) and tests on the held-out 1/5.
Every image gets exactly one out-of-fold prediction.

In [ ]:
strata = (df["label"].astype(str) + "_" + df["source"]).to_numpy()
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X, strata))

oof_prob = np.full(len(df), np.nan)
oof_thr = np.full(len(df), np.nan)
fold_rows = []

for k, (trval, te) in enumerate(FOLDS):
    print(f"\n==================== FOLD {k + 1}/{N_FOLDS} ====================")
    tr, va = train_test_split(trval, test_size=INNER_VAL_FRACTION, stratify=strata[trval], random_state=SEED)
    model, _ = train_model(tr, va)

    thr = pick_threshold(y[va], predict(model, va))
    p_te = predict(model, te)
    oof_prob[te], oof_thr[te] = p_te, thr

    row = {"fold": k + 1, **summarize(y[te], p_te, thr)}
    fold_rows.append(row)
    print({kk: round(v, 4) if isinstance(v, float) else v for kk, v in row.items()})
    model.save(os.path.join(OUT_DIR, f"{BACKBONE}_fold{k + 1}.keras"))

fold_df = pd.DataFrame(fold_rows)
display(fold_df.round(4))
print("\nMean +/- std across folds:")
display(fold_df[["auc", "sensitivity", "specificity", "accuracy"]].agg(["mean", "std"]).round(4))

## Out-of-fold results (overall and per source)

In [ ]:
oof_pred = (oof_prob >= oof_thr).astype(int)
lo, hi = bootstrap_auc_ci(y, oof_prob)
overall = summarize(y, oof_prob, 0.5) | {"threshold": "per-fold"}
tn, fp, fn, tp = confusion_matrix(y, oof_pred, labels=[0, 1]).ravel()
overall.update(sensitivity=tp / (tp + fn), specificity=tn / (tn + fp), accuracy=(tp + tn) / len(y))
print(f"OOF AUC = {overall['auc']:.4f}  (95% CI {lo:.3f}-{hi:.3f})")
print(f"At the per-fold thresholds (target sensitivity {TARGET_SENSITIVITY:.0%}): "
      f"sensitivity={overall['sensitivity']:.3f}  specificity={overall['specificity']:.3f}  "
      f"accuracy={overall['accuracy']:.3f}")

per_source = []
for src in df["source"].unique():
    m = (df["source"] == src).to_numpy()
    t_n, f_p, f_n, t_p = confusion_matrix(y[m], oof_pred[m], labels=[0, 1]).ravel()
    per_source.append(dict(source=src, n=int(m.sum()), auc=roc_auc_score(y[m], oof_prob[m]),
                           sensitivity=t_p / (t_p + f_n), specificity=t_n / (t_n + f_p)))
display(pd.DataFrame(per_source).round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for src in ["all"] + list(df["source"].unique()):
    m = np.ones(len(df), bool) if src == "all" else (df["source"] == src).to_numpy()
    fpr, tpr, _ = roc_curve(y[m], oof_prob[m])
    axes[0].plot(fpr, tpr, label=f"{src} (AUC {roc_auc_score(y[m], oof_prob[m]):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].set(xlabel="False positive rate", ylabel="Sensitivity", title="Out-of-fold ROC")
axes[0].legend()
cm = confusion_matrix(y, oof_pred, labels=[0, 1])
axes[1].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    axes[1].text(j, i, v, ha="center", va="center", fontsize=14)
axes[1].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Normal", "TB"], yticklabels=["Normal", "TB"],
            xlabel="Predicted", ylabel="True", title="Out-of-fold confusion matrix")
plt.tight_layout(); plt.show()

## Cross-source test: how well does it generalise?
This trains on one dataset and tests on the other. Expect a drop compared with the CV numbers. That drop is the
honest estimate of how the model will do on X-rays from a new hospital or scanner.

In [ ]:
cross_rows = []
if USE_SECOND_DATASET and RUN_CROSS_SOURCE:
    for train_src, test_src in [("shenzhen", "dataset2"), ("dataset2", "shenzhen")]:
        print(f"\n==================== TRAIN {train_src} -> TEST {test_src} ====================")
        src_idx = np.where(df["source"] == train_src)[0]
        te = np.where(df["source"] == test_src)[0]
        tr, va = train_test_split(src_idx, test_size=INNER_VAL_FRACTION, stratify=y[src_idx], random_state=SEED)
        model, _ = train_model(tr, va)
        thr = pick_threshold(y[va], predict(model, va))
        cross_rows.append({"train": train_src, "test": test_src, **summarize(y[te], predict(model, te), thr)})
    display(pd.DataFrame(cross_rows).round(4))

## Grad-CAM: where is the model looking?
Uses the fold-1 model on fold-1 test images (images it never trained on). Heatmaps should sit inside the lungs,
mostly the upper zones for TB. Heat on text markers, image borders or outside the chest points to shortcut learning.

In [ ]:
def find_backbone(model):
    for layer in model.layers:
        if isinstance(layer, keras.Model):
            try:
                layer.get_layer(CFG["last_conv"])
                return layer
            except ValueError:
                pass
    raise ValueError("backbone not found")


def gradcam(model, img_uint8):
    backbone = find_backbone(model)
    conv_model = keras.Model(backbone.inputs, [backbone.get_layer(CFG["last_conv"]).output, backbone.output])
    x = np.repeat(img_uint8[None, ..., None], 3, axis=-1).astype("float32")
    with tf.GradientTape() as tape:
        z = model.get_layer("preprocess")(model.get_layer("clip")(x))
        conv, feats = conv_model(z, training=False)
        prob = model.get_layer("head")(model.get_layer("gap")(feats))[:, 0]
    grads = tape.gradient(prob, conv)
    weights = tf.reduce_mean(tf.cast(grads, tf.float32), axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(tf.cast(conv, tf.float32) * weights[:, None, None, :], axis=-1))[0].numpy()
    cam = cv2.resize(cam / (cam.max() + 1e-8), (IMG_SIZE, IMG_SIZE))
    return cam, float(prob[0])


cam_model = keras.models.load_model(os.path.join(OUT_DIR, f"{BACKBONE}_fold1.keras"))
te0 = FOLDS[0][1]
rng = np.random.default_rng(SEED)
show = np.concatenate([rng.choice(te0[y[te0] == 1], 4, replace=False), rng.choice(te0[y[te0] == 0], 4, replace=False)])
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axes.ravel(), show):
    cam, p = gradcam(cam_model, X[i])
    ax.imshow(X[i], cmap="gray"); ax.imshow(cam, cmap="jet", alpha=0.35)
    ax.set_title(f"{'TB' if y[i] else 'Normal'} | P(TB)={p:.2f} | {df.source[i]}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## Inference: 5-fold ensemble
This averages the 5 fold models. The decision threshold is the median of the per-fold thresholds.

In [ ]:
fold_models = [keras.models.load_model(p)
               for p in sorted(glob.glob(os.path.join(OUT_DIR, f"{BACKBONE}_fold*.keras")))]
THRESHOLD = float(np.median(fold_df["threshold"]))
print(f"Loaded {len(fold_models)} models | decision threshold = {THRESHOLD:.3f}")


def predict_images(images, names=None):
    """images: list of file paths or raw bytes."""
    arr = np.stack([load_gray(im) for im in images])
    x = np.repeat(arr[..., None], 3, axis=-1).astype("float32")
    prob = np.mean([m.predict(x, verbose=0).ravel() for m in fold_models], axis=0)
    names = names or [os.path.basename(im) if isinstance(im, str) else f"image_{i}" for i, im in enumerate(images)]
    return pd.DataFrame({"file": names, "tb_probability": prob.round(4),
                         "prediction": np.where(prob >= THRESHOLD, "TB", "Normal")})


def predict_upload(upload_widget):
    """Works with the ipywidgets FileUpload widget (v7 and v8)."""
    files = list(upload_widget.value) if isinstance(upload_widget.value, (tuple, list)) \
        else list(upload_widget.value.values())
    data = [bytes(f["content"]) for f in files]
    names = [f.get("name", f"image_{i}") for i, f in enumerate(files)]
    return predict_images(data, names)


display(predict_images(df["image_path"].iloc[:5].tolist()))

## Optional: TorchXRayVision baseline (frozen chest X-ray features + logistic regression)
This uses the same folds, so the AUC is directly comparable with the InceptionV3 result above. It runs in a few
minutes. It needs internet enabled in Kaggle settings, because the weights download from GitHub.

In [ ]:
%pip install -q torchxrayvision

In [ ]:
import torch
import torchxrayvision as xrv
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

device = "cuda" if torch.cuda.is_available() else "cpu"
xrv_model = xrv.models.DenseNet(weights="densenet121-res224-all").to(device).eval()

feats = []
with torch.no_grad():
    for s in range(0, len(X), 32):
        batch = np.stack([cv2.resize(im, (224, 224), interpolation=cv2.INTER_AREA) for im in X[s:s + 32]])
        batch = xrv.datasets.normalize(batch.astype(np.float32), 255)[:, None]  # xrv expects [-1024, 1024]
        feats.append(xrv_model.features2(torch.from_numpy(batch).to(device)).cpu().numpy())
feats = np.concatenate(feats)
print("TorchXRayVision features:", feats.shape)

xrv_oof = np.full(len(df), np.nan)
for trval, te in FOLDS:
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=5000))
    clf.fit(feats[trval], y[trval])
    xrv_oof[te] = clf.predict_proba(feats[te])[:, 1]

comparison = pd.DataFrame([
    {"model": f"{BACKBONE} fine-tuned", "oof_auc": roc_auc_score(y, oof_prob)},
    {"model": "TorchXRayVision frozen + LogReg", "oof_auc": roc_auc_score(y, xrv_oof)},
    {"model": "Average of both", "oof_auc": roc_auc_score(y, (oof_prob + xrv_oof) / 2)},
])
for src in df["source"].unique():
    m = (df["source"] == src).to_numpy()
    comparison[f"auc_{src}"] = [roc_auc_score(y[m], oof_prob[m]), roc_auc_score(y[m], xrv_oof[m]),
                                roc_auc_score(y[m], ((oof_prob + xrv_oof) / 2)[m])]
display(comparison.round(4))